# Niural support agent - v7

Account tools, a batched Decimal calculator and hybrid product-guide retrieval.
Models and optional LangSmith tracing are configured in the repository-root `.env`.
Use the CLI for streaming chat; use this notebook for manually enabled evaluation.
Answer generation, judging and batch evaluation are off by default.


## 1. Setup
Select the **Conda base** kernel (`C:/Users/Nirvana/miniconda3/python.exe`). `OPENAI_API_KEY` is loaded from the project `.env`.
This cell creates the agent without making a model request.


In [ ]:
from pathlib import Path
import json
import sys
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "mock_api/data").is_dir() and (p / "niural_support_agent/config.py").is_file())
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from niural_support_agent.config import Config, BENCHMARK_DIR
from mock_api import MockAPI, APIError
from niural_support_agent.agent import SupportAgent
from niural_support_agent.evaluation.account import (evaluate_one, evaluate_correctness,
                                           case_inputs, case_references)

config = Config.load()
agent = SupportAgent(config)
display({"provider": config.provider, "model": config.model, "judge_model": config.judge_model})


## 2. Mock API and token auth
`mock_api/` contains the client, auth, and Pydantic data models. Any nonempty
mock token is accepted; a missing token raises a 401 API error. This token is separate from the OpenAI key.


In [ ]:
api = MockAPI(token=config.mock_token)
employees = api.list_employees()
display({"employee_records": len(employees.data), "source": employees.source.model_dump(mode="json")})

try:
    MockAPI(token="").list_vendors()
except APIError as error:
    display(error.as_dict())

display([tool.name for tool in agent.agent.tools])


### Generated function schema (no model request)
The SDK creates this definition from the function signature and docstring. All four tools are available upfront; no deferred loading.


In [ ]:
tool = next(t for t in agent.agent.tools)
display({"type": "function", "name": tool.name, "description": tool.description,
         "parameters": tool.params_json_schema, "strict": tool.strict_json_schema})


### Local query example (no model request)
This exercises the same validated query and Python executor used by the tool.
Counts come from code; the model only chooses the query and explains its result.


In [ ]:
from niural_support_agent.tools.query import AccountQuery, query_records

query = AccountQuery(
    resource="employees", operation="count",
    filters=[{"field": "status", "operator": "eq", "value": "ONBOARDED"}],
    group_by=["employment_type"],
)
display(query_records(api.list_employees(), query).model_dump(mode="json"))


## 3. Streaming chat
Run this cell, then type a message. Each message starts one agent turn; text appears as it arrives.
`/reset` clears this chat, `/quit` exits. No judge calls are made.


In [ ]:
chat_session = "notebook-chat"
print("Chat ready. /reset clears history; /quit exits.")

while True:
    try:
        question = input("You: ").strip()
    except EOFError:
        break
    if question.lower() == "/quit":
        break
    if question.lower() == "/reset":
        agent.reset(chat_session)
        print("Conversation cleared.")
        continue
    if not question:
        continue

    print("Assistant: ", end="", flush=True)
    chunks = []
    def show_text(text):
        chunks.append(text)
        print(text, end="", flush=True)

    response = await agent.ask(chat_session, question, on_text=show_text)
    if response.error:
        print("\n" + (response.error.get("detail") or response.error["message"]))
    elif not chunks:
        print(response.answer, end="")
    print("\n")


Chat ready. /reset clears history; /quit exits.


## 4. Select one benchmark question
Choose `suite`: `standard`, `hard` or `arithmetic`, and `case_id` from 1 to 8.
Compare the same arithmetic cases against v2; v4 adds the calculator; keep the cases and evaluation settings unchanged.
Reference facts are used only by the optional judge.


In [ ]:
from niural_support_agent.evaluation.run_benchmark import SUITES, run_benchmark

suite = "arithmetic"
case_id = 1
filename, _ = SUITES[suite]
cases = json.loads((BENCHMARK_DIR / filename).read_text(encoding="utf-8"))
case = next(row for row in cases if row["id"] == case_id)
print(case["question"])


## 5. Generate one benchmark answer
Set `RUN_ANSWER = True` when ready. One question may require several model/tool calls,
up to `config.max_turns` model turns. A rate-limit error stops the run; there are no automatic retries.


In [ ]:
RUN_ANSWER = False
result = None
if RUN_ANSWER:
    result = await evaluate_one(agent, case)
    display(Markdown(result["answer"]) if result["answer"] else result["error"])
    display(result["usage"])
else:
    print("Answer generation is off. Set RUN_ANSWER = True to run this question.")


## 6. Optional correctness judge
This makes an additional model request. Leave it off to save quota.
The answer is compared with the verified reference facts using the shared rubric.


In [ ]:
RUN_JUDGE = False
if RUN_JUDGE and result and result["id"] == case["id"]:
    verdict = await evaluate_correctness(
        agent, case_inputs(case), result, case_references(case))
    display(verdict)
else:
    print("Judge is off. Enable it explicitly after generating an answer.")


## 7. Run all eight in LangSmith (optional)
Restart the kernel and rerun setup after enabling LangSmith in the repository-root `.env`.
This uses the selected `suite`. Set RUN_ALL to True only when ready: it runs eight answers
plus one judge per successful answer. Default concurrency is one.
Set judge=False to save answers/traces without paid judge calls.


In [ ]:
RUN_ALL = False
if RUN_ALL:
    experiment = await run_benchmark(
        agent, name=f"v4-{suite}-calculator",
        description="V4: batched Decimal calculator with explicit rounding",
        max_concurrency=1, judge=True, suite=suite,
    )
    display(experiment.to_pandas())
else:
    print("Batch evaluation is off. Set RUN_ALL = True to run the selected suite.")


## 7. V4 observations

- What worked:
- What failed:
- One proposed change:
- Retain / rollback / inconclusive:

Results remain in notebook outputs. Nothing is automatically saved to run folders.
